In [22]:
import os
import shutil
import zipfile
import tempfile

import ipywidgets as widgets
from IPython.display import display, clear_output


# ----------------------------
# UI
# ----------------------------

L = widgets.BoundedFloatText(
    value=100,
    min=1,
    max=100000,
    step=10,
    description="Width:"
)

H = widgets.BoundedFloatText(
    value=100,
    min=1,
    max=100000,
    step=10,
    description="Height:"
)

dim = widgets.HBox([L, H])

file_upload = widgets.FileUpload(
    accept=".jpg,.jpeg,.png,.bmp,.tif,.tiff",
    multiple=False,
    description="Select Image"
)

convert_button = widgets.Button(
    description="Create Model",
    button_style="success"
)

output = widgets.Output()


# ----------------------------
# OBJ Creator
# ----------------------------

def create_textured_obj(
    image_path,
    image_name,
    width,
    height
):
    base_name = os.path.splitext(image_name)[0]

    obj_file = os.path.abspath(base_name + ".obj")
    mtl_file = os.path.abspath(base_name + ".mtl")

    texture_name = os.path.basename(image_name)

    # Copy texture beside OBJ
    texture_copy = os.path.abspath(texture_name)
    shutil.copy2(image_path, texture_copy)

    # OBJ
    with open(obj_file, "w") as f:

        f.write(f"mtllib {os.path.basename(mtl_file)}\n")
        f.write("usemtl imageMat\n\n")

        # vertices
        f.write(f"v 0 0 0\n")
        f.write(f"v {width} 0 0\n")
        f.write(f"v {width} {height} 0\n")
        f.write(f"v 0 {height} 0\n\n")

        # UV coordinates
        f.write("vt 0 0\n")
        f.write("vt 1 0\n")
        f.write("vt 1 1\n")
        f.write("vt 0 1\n\n")

        # normal
        f.write("vn 0 0 1\n\n")

        # triangles
        f.write("f 1/1/1 2/2/1 3/3/1\n")
        f.write("f 1/1/1 3/3/1 4/4/1\n")

    # MTL
    with open(mtl_file, "w") as f:

        f.write("newmtl imageMat\n")
        f.write("Ka 1.000 1.000 1.000\n")
        f.write("Kd 1.000 1.000 1.000\n")
        f.write("Ks 0.000 0.000 0.000\n")
        f.write("d 1.0\n")
        f.write("illum 2\n")
        f.write(f"map_Kd {texture_name}\n")

    # ZIP package
    zip_file = os.path.abspath(base_name + ".zip")

    with zipfile.ZipFile(
        zip_file,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:

        z.write(obj_file, os.path.basename(obj_file))
        z.write(mtl_file, os.path.basename(mtl_file))
        z.write(texture_copy, texture_name)

    return {
        "obj": obj_file,
        "mtl": mtl_file,
        "texture": texture_copy,
        "zip": zip_file
    }


# ----------------------------
# Button Event
# ----------------------------

def convert_clicked(btn):

    with output:

        clear_output()

        if not file_upload.value:
            print("Please select an image.")
            return

        uploaded = file_upload.value[0]

        original_name = uploaded["name"]

        ext = os.path.splitext(original_name)[1]

        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=ext
        ) as temp_file:

            temp_file.write(uploaded["content"])
            image_path = temp_file.name

        try:

            result = create_textured_obj(
                image_path=image_path,
                image_name=original_name,
                width=L.value,
                height=H.value
            )

            print("✅ Model Created\n")

            print("OBJ:")
            print(result["obj"])

            print("\nMTL:")
            print(result["mtl"])

            print("\nTexture:")
            print(result["texture"])

            print("\nZIP Package:")
            print(result["zip"])

        except Exception as ex:

            print(f"❌ Error: {ex}")


convert_button.on_click(convert_clicked)


# ----------------------------
# Display UI
# ----------------------------

display(
    widgets.VBox([
        dim,
        file_upload,
        convert_button,
        output
    ])
)